### NumPy Exercises: Supermarket Sales

#### 📋 Dataset Information

**Dataset**: Supermarket Sales  
**Source**: Attached file  

**Columns**:
- `Branch`: Store branch identifier where the transaction occurred ('A', 'B', or 'C').
- `Customer type`: Customer membership status ('Member', 'Normal').
- `Gende`r: Customer's gender ('Male' or 'Female').
- `Product line`: Product category purchased (6 categories: Electronic accessories, Fashion accessories, Food and beverages, Health and beauty, Home and lifestyle, Sports and travel).
- `Quantity`: Number of items purchased in the transaction.
- `Total`: Total amount paid by customer in dollars, including tax.
- `Date`: Transaction date in M/D/YYYY format.
- `Rating`: Customer satisfaction rating on a scale of 0.0 to 10.0 (higher is better).

In [1]:
import numpy as np

In [2]:
## NO NEED CODE HERE
# Load the CSV file
data = np.genfromtxt(
    'supermarket_sales.csv',
    delimiter=',',
    skip_header=1,    # Skip header row
    dtype=str         # Load as strings (no preprocessing)
)

# Print basic info
print("="*60)
print("DATASET LOADED")
print("="*60)
print(f"Shape: {data.shape}")
print(f"  Rows: {data.shape[0]} transactions")
print(f"  Columns: {data.shape[1]}")

# Column names
columns = ['Branch', 'Customer type', 'Gender', 'Product line', 
           'Quantity', 'Total', 'Date', 'Rating']

print(f"\nColumns:")
for i, col in enumerate(columns):
    print(f"  [{i}] {col}")

DATASET LOADED
Shape: (1000, 8)
  Rows: 1000 transactions
  Columns: 8

Columns:
  [0] Branch
  [1] Customer type
  [2] Gender
  [3] Product line
  [4] Quantity
  [5] Total
  [6] Date
  [7] Rating


### Task 1: Customer Segmentation Analysis
**Which customer segment (by type and gender) generates the highest revenue per transaction, and how does their rating behavior differ from other segments?** 

**Instruction:**
- 4 customer segments: Member-Female, Member-Male, Normal-Female, Normal-Male.
- Calculate revenue per transaction for each segment.
- Calculate average rating for each segment.
- How each segment compares to the overall average?
   - Calculate deviations 
- Which segment is the "best"? (Based on revenue, rating and count)
   - Weight: 50% revenue, 30% rating, 20% count (volume) (Need normalize to [0.1])

In [3]:
#TODO: your code here

# Extract relevant columns
customer_type = data[:, 1]
gender = data[:, 2]
total = data[:, 5].astype(float)
rating = data[:, 7].astype(float)

# Define the four customer segments
segments = {
    'Member-Female': (customer_type == 'Member') & (gender == 'Female'),
    'Member-Male': (customer_type == 'Member') & (gender == 'Male'),
    'Normal-Female': (customer_type == 'Normal') & (gender == 'Female'),
    'Normal-Male': (customer_type == 'Normal') & (gender == 'Male')
}

# Calculate overall averages
overall_avg_revenue = np.mean(total)
overall_avg_rating = np.mean(rating)

# Analyze each segment
results = {}
for name, mask in segments.items():
    count = np.sum(mask)
    if count > 0:
        avg_revenue = np.mean(total[mask])
        avg_rating = np.mean(rating[mask])
    else:
        avg_revenue = 0
        avg_rating = 0
    
    results[name] = {
        'count': count,
        'avg_revenue': avg_revenue,
        'avg_rating': avg_rating,
        'revenue_deviation': avg_revenue - overall_avg_revenue,
        'rating_deviation': avg_rating - overall_avg_rating
    }

# Normalize metrics for scoring
counts = np.array([res['count'] for res in results.values()])
revenues = np.array([res['avg_revenue'] for res in results.values()])
ratings = np.array([res['avg_rating'] for res in results.values()])

norm_counts = (counts - np.min(counts)) / (np.max(counts) - np.min(counts))
norm_revenues = (revenues - np.min(revenues)) / (np.max(revenues) - np.min(revenues))
norm_ratings = (ratings - np.min(ratings)) / (np.max(ratings) - np.min(ratings))

# Calculate the final score for each segment
scores = 0.2 * norm_counts + 0.5 * norm_revenues + 0.3 * norm_ratings
best_segment_index = np.argmax(scores)
best_segment_name = list(results.keys())[best_segment_index]

# Print the results
print('--- Customer Segmentation Analysis ---')
for i, (name, res) in enumerate(results.items()):
    print(f"\nSegment: {name}")
    print(f"  - Transactions: {res['count']}")
    print(f"  - Avg Revenue: ${res['avg_revenue']:.2f} (Deviation: ${res['revenue_deviation']:.2f})")
    print(f"  - Avg Rating: {res['avg_rating']:.2f} (Deviation: {res['rating_deviation']:.2f})")
    print(f"  - Normalized Score: {scores[i]:.4f}")

print(f"\n--- Best Performing Segment ---")
print(f"The best customer segment is '{best_segment_name}' with a score of {scores[best_segment_index]:.4f}.")

--- Customer Segmentation Analysis ---

Segment: Member-Female
  - Transactions: 261
  - Avg Revenue: $337.73 (Deviation: $14.76)
  - Avg Rating: 6.94 (Deviation: -0.03)
  - Normalized Score: 0.7023

Segment: Member-Male
  - Transactions: 240
  - Avg Revenue: $316.99 (Deviation: $-5.98)
  - Avg Rating: 6.94 (Deviation: -0.03)
  - Normalized Score: 0.1826

Segment: Normal-Female
  - Transactions: 240
  - Avg Revenue: $332.23 (Deviation: $9.27)
  - Avg Rating: 6.99 (Deviation: 0.02)
  - Normalized Score: 0.6076

Segment: Normal-Male
  - Transactions: 259
  - Avg Revenue: $305.05 (Deviation: $-17.92)
  - Avg Rating: 7.02 (Deviation: 0.05)
  - Normalized Score: 0.4810

--- Best Performing Segment ---
The best customer segment is 'Member-Female' with a score of 0.7023.


### Task 2: Product Performance Optimization
**Which product lines are underperforming (below average sales) in which branches, and what is the revenue opportunity if they reached branch-average performance?**

**Instructions**:
1. **18 combinations**: 3 Branches × 6 Product Lines = 18
2. **Average sales per product-branch combination**
3. **Identify underperformers**: Which are below their branch average?
4. **Calculate gap**: How much below average?
5. **Revenue opportunity**: Potential gain if they reached branch average

In [4]:
#TODO: your code here

# Extract relevant columns
branch = data[:, 0]
product_line = data[:, 3]
total = data[:, 5].astype(float)

# Get unique branches and product lines
unique_branches = np.unique(branch)
unique_product_lines = np.unique(product_line)

print('--- Product Performance Optimization ---')

# Analyze each branch
for b in unique_branches:
    branch_mask = (branch == b)
    branch_total_revenue = np.sum(total[branch_mask])
    branch_transactions = np.sum(branch_mask)
    branch_avg_revenue = branch_total_revenue / branch_transactions
    
    print(f"\n--- Branch: {b} (Avg. Revenue: ${branch_avg_revenue:.2f}) ---")
    
    # Analyze each product line within the branch
    for pl in unique_product_lines:
        product_mask = (product_line == pl)
        combination_mask = branch_mask & product_mask
        
        combination_transactions = np.sum(combination_mask)
        
        if combination_transactions > 0:
            combination_avg_revenue = np.sum(total[combination_mask]) / combination_transactions
            
            # Check if it's underperforming
            if combination_avg_revenue < branch_avg_revenue:
                gap = branch_avg_revenue - combination_avg_revenue
                opportunity = gap * combination_transactions
                print(f"  - [UNDERPERFORMING] Product Line: {pl}")
                print(f"    - Avg. Revenue: ${combination_avg_revenue:.2f}")
                print(f"    - Performance Gap: ${gap:.2f}")
                print(f"    - Revenue Opportunity: ${opportunity:.2f}")
            else:
                print(f"  - [OK] Product Line: {pl} (Avg. Revenue: ${combination_avg_revenue:.2f})")
        else:
            print(f"  - [N/A] Product Line: {pl} (No transactions)")

--- Product Performance Optimization ---

--- Branch: A (Avg. Revenue: $312.35) ---
  - [UNDERPERFORMING] Product Line: Electronic accessories
    - Avg. Revenue: $305.29
    - Performance Gap: $7.07
    - Revenue Opportunity: $424.13
  - [OK] Product Line: Fashion accessories (Avg. Revenue: $320.25)
  - [UNDERPERFORMING] Product Line: Food and beverages
    - Avg. Revenue: $295.92
    - Performance Gap: $16.44
    - Revenue Opportunity: $953.43
  - [UNDERPERFORMING] Product Line: Health and beauty
    - Avg. Revenue: $268.04
    - Performance Gap: $44.32
    - Revenue Opportunity: $2082.89
  - [OK] Product Line: Home and lifestyle (Avg. Revenue: $344.88)
  - [OK] Product Line: Sports and travel (Avg. Revenue: $328.35)

--- Branch: B (Avg. Revenue: $319.87) ---
  - [UNDERPERFORMING] Product Line: Electronic accessories
    - Avg. Revenue: $310.03
    - Performance Gap: $9.85
    - Revenue Opportunity: $541.54
  - [UNDERPERFORMING] Product Line: Fashion accessories
    - Avg. Revenue: $

### Task 3: High-Value Customer Identification
**What percentage of total revenue comes from the top 20% of transactions, and what are the common characteristics of these high-value transactions?**

**Instructions**:\
**1. Identify the Top 20% of Transactions by Total amount**

**2. Calculate Revenue Concentration**\
Determine what percentage of total revenue is generated by these top 20% of transactions.

**3. Profile High-Value Transaction Characteristics**\
For the top 20% transactions, analyze their common patterns across multiple dimensions. How many transaction happen: 
 - On each branches
 - On each product lines
 - On each customer types
 - On each gender

**4. Compare High-Value vs Overall Distribution**\
Calculate how the characteristics of high-value transactions differ from the overall dataset by comparing percentage distributions across branches, products, customer types, and gender. \
For example, if Branch A represents 40% of high-value transactions but only 33% of all transactions, this indicates Branch A has a premium customer base. 

**5. Calculate Average Metrics for High-Value Segment**\
Compute the average transaction amount, average quantity purchased, and average rating specifically for the top 20% group and compare these averages to the overall dataset averages. For example:
- Top 20% spend 125% MORE per transaction
- Top 20% buy 55% MORE items per transaction
- Top 20% Ratings ....
  
**6. Identify the "Golden Combination"**\
Find the most common combination of characteristics (Branch + Product + Customer Type + Gender) within the high-value segment

**7. Analyze Contribution by Percentile Groups**\
Beyond just the top 20%, how revenue is distributed across all percentile groups (top 20%, 20-40%, 40-60%, 60-80%, bottom 20%) 

In [5]:
#TODO: your code here

# --- 1. Identify Top 20% of Transactions ---
total = data[:, 5].astype(float)
percentile_80 = np.percentile(total, 80)
top_20_mask = total >= percentile_80
top_20_data = data[top_20_mask]

# --- 2. Calculate Revenue Concentration ---
total_revenue = np.sum(total)
top_20_revenue = np.sum(total[top_20_mask])
revenue_concentration = (top_20_revenue / total_revenue) * 100
print(f'--- High-Value Customer Identification ---')
print(f'The top 20% of transactions account for {revenue_concentration:.2f}% of total revenue.')

# --- 3. Profile High-Value Transaction Characteristics ---
print('\n--- Profile of Top 20% Transactions ---')
characteristics = ['Branch', 'Product line', 'Customer type', 'Gender']
column_indices = [0, 3, 1, 2]
for char, idx in zip(characteristics, column_indices):
    overall_values, overall_counts = np.unique(data[:, idx], return_counts=True)
    top_20_values, top_20_counts = np.unique(top_20_data[:, idx], return_counts=True)
    
    print(f'\nDistribution by {char}:')
    for i, val in enumerate(overall_values):
        overall_percentage = (overall_counts[i] / len(data)) * 100
        top_20_idx = np.where(top_20_values == val)[0]
        if len(top_20_idx) > 0:
            top_20_percentage = (top_20_counts[top_20_idx[0]] / len(top_20_data)) * 100
        else:
            top_20_percentage = 0
        print(f'  - {val}: {top_20_percentage:.2f}% (Top 20%) vs {overall_percentage:.2f}% (Overall)')

# --- 5. Calculate Average Metrics for High-Value Segment ---
print('\n--- Average Metrics Comparison ---')
quantity = data[:, 4].astype(int)
rating = data[:, 7].astype(float)

overall_avg_total = np.mean(total)
top_20_avg_total = np.mean(total[top_20_mask])
total_increase = ((top_20_avg_total - overall_avg_total) / overall_avg_total) * 100

overall_avg_quantity = np.mean(quantity)
top_20_avg_quantity = np.mean(quantity[top_20_mask])
quantity_increase = ((top_20_avg_quantity - overall_avg_quantity) / overall_avg_quantity) * 100

overall_avg_rating = np.mean(rating)
top_20_avg_rating = np.mean(rating[top_20_mask])
rating_diff = top_20_avg_rating - overall_avg_rating

print(f'- Top 20% spend {total_increase:.2f}% MORE per transaction.')
print(f'- Top 20% buy {quantity_increase:.2f}% MORE items per transaction.')
print(f'- Top 20% ratings are {abs(rating_diff):.2f} points {'higher' if rating_diff > 0 else 'lower'} on average.')

# --- 6. Identify the \"Golden Combination\" ---
print('\n--- Golden Combination in Top 20% ---')
combinations = np.char.add(np.char.add(np.char.add(top_20_data[:, 0], ' - '), top_20_data[:, 3]), ' - ')
combinations = np.char.add(np.char.add(np.char.add(combinations, top_20_data[:, 1]), ' - '), top_20_data[:, 2])
unique_combinations, counts = np.unique(combinations, return_counts=True)
golden_combination = unique_combinations[np.argmax(counts)]
print(f'The most frequent combination is: "{golden_combination}"')

# --- 7. Analyze Contribution by Percentile Groups ---
print('\n--- Revenue Distribution by Quintiles ---')
quintiles = [20, 40, 60, 80, 100]
percentile_values = np.percentile(total, quintiles)
last_percentile = 0

for i, p in enumerate(percentile_values):
    if i == 0:
        mask = total <= p
        label = f'Bottom 20%'
    else:
        mask = (total > last_percentile) & (total <= p)
        label = f'{quintiles[i-1]}-{quintiles[i]}%'
        
    revenue_in_quintile = np.sum(total[mask])
    contribution = (revenue_in_quintile / total_revenue) * 100
    print(f'- {label}: {contribution:.2f}% of total revenue')
    last_percentile = p

--- High-Value Customer Identification ---
The top 20% of transactions account for 45.08% of total revenue.

--- Profile of Top 20% Transactions ---

Distribution by Branch:
  - A: 30.00% (Top 20%) vs 34.00% (Overall)
  - B: 33.50% (Top 20%) vs 33.20% (Overall)
  - C: 36.50% (Top 20%) vs 32.80% (Overall)

Distribution by Product line:
  - Electronic accessories: 18.50% (Top 20%) vs 17.00% (Overall)
  - Fashion accessories: 15.50% (Top 20%) vs 17.80% (Overall)
  - Food and beverages: 15.50% (Top 20%) vs 17.40% (Overall)
  - Health and beauty: 16.00% (Top 20%) vs 15.20% (Overall)
  - Home and lifestyle: 16.50% (Top 20%) vs 16.00% (Overall)
  - Sports and travel: 18.00% (Top 20%) vs 16.60% (Overall)

Distribution by Customer type:
  - Member: 52.00% (Top 20%) vs 50.10% (Overall)
  - Normal: 48.00% (Top 20%) vs 49.90% (Overall)

Distribution by Gender:
  - Female: 52.00% (Top 20%) vs 50.10% (Overall)
  - Male: 48.00% (Top 20%) vs 49.90% (Overall)

--- Average Metrics Comparison ---
- Top 2